# Homework 10_11 — Salary Regression (with optional derived classification)

This notebook adapts core model-development demonstrations to the `Salary_dataset.csv` dataset.

Main task: **predict `Salary` (continuous target) from `YearsExperience`**.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('default')


## 1) Load data robustly and inspect it

In [ ]:
def load_salary_dataset() -> tuple[pd.DataFrame, Path]:
    """Load Salary_dataset.csv from common repository-relative locations."""
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    candidates = []
    for root in search_roots:
        candidates.extend([
            root / 'Salary_dataset.csv',
            root / 'data' / 'Salary_dataset.csv',
            root / 'dataset' / 'Salary_dataset.csv',
            root / 'datasets' / 'Salary_dataset.csv',
        ])

    seen = set()
    ordered_candidates = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            ordered_candidates.append(c)

    for path in ordered_candidates:
        if path.exists():
            return pd.read_csv(path), path

    recursive_hits = list(Path.cwd().glob('**/Salary_dataset.csv'))
    if recursive_hits:
        return pd.read_csv(recursive_hits[0]), recursive_hits[0]

    searched = '\n'.join(str(p.resolve()) for p in ordered_candidates)
    raise FileNotFoundError(
        "Could not find Salary_dataset.csv. Searched:\n" + searched
    )


df_raw, data_path = load_salary_dataset()
print(f'Loaded: {data_path}')
print(f'Shape: {df_raw.shape}')
df_raw.head()


In [ ]:
# Ignore CSV index column if present (typical: 'Unnamed: 0')
index_like_cols = [c for c in df_raw.columns if str(c).lower().startswith('unnamed')]
df = df_raw.drop(columns=index_like_cols, errors='ignore').copy()

print('Columns after cleanup:', list(df.columns))
print('\nMissing values per column:')
print(df.isna().sum())

required_cols = ['YearsExperience', 'Salary']
df = df.dropna(subset=required_cols).copy()

print(f'Rows after required-column cleanup: {len(df)}')
print(df.describe())


In [ ]:
X = df[['YearsExperience']].copy()   # Feature
y = df['Salary'].copy()              # Continuous target

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X['YearsExperience'], y, alpha=0.8)
ax.set_title('Salary vs YearsExperience')
ax.set_xlabel('YearsExperience')
ax.set_ylabel('Salary')
plt.show()


## 2) Polynomial fitting + overfitting / bias-variance behavior

In [ ]:
plot_degrees = [1, 2, 5, 9]
x_plot = np.linspace(X['YearsExperience'].min(), X['YearsExperience'].max(), 200).reshape(-1, 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, deg in zip(axes, plot_degrees):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lin', LinearRegression())
    ])
    model.fit(X, y)
    y_plot = model.predict(x_plot)

    ax.scatter(X['YearsExperience'], y, alpha=0.65, label='data')
    ax.plot(x_plot.ravel(), y_plot, color='crimson', lw=2, label=f'degree={deg}')
    ax.set_title(f'Polynomial fit (degree={deg})')
    ax.set_xlabel('YearsExperience')
    ax.set_ylabel('Salary')
    ax.legend(loc='best')

plt.tight_layout()
plt.show()


In [ ]:
max_degree = 12
degrees = list(range(1, max_degree + 1))
repeats = 80

train_rmse_means = []
test_rmse_means = []
train_rmse_stds = []
test_rmse_stds = []

for deg in degrees:
    tr_errors, te_errors = [], []
    for seed in range(repeats):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=seed
        )
        model = Pipeline([
            ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
            ('lin', LinearRegression())
        ])
        model.fit(X_train, y_train)
        y_tr_pred = model.predict(X_train)
        y_te_pred = model.predict(X_test)
        tr_errors.append(np.sqrt(mean_squared_error(y_train, y_tr_pred)))
        te_errors.append(np.sqrt(mean_squared_error(y_test, y_te_pred)))

    train_rmse_means.append(np.mean(tr_errors))
    test_rmse_means.append(np.mean(te_errors))
    train_rmse_stds.append(np.std(tr_errors))
    test_rmse_stds.append(np.std(te_errors))

best_idx = int(np.argmin(test_rmse_means))
best_degree = degrees[best_idx]
print(f'Best average test RMSE at degree={best_degree}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees, train_rmse_means, marker='o', label='Train RMSE (mean)')
ax.plot(degrees, test_rmse_means, marker='s', label='Test RMSE (mean)')
ax.fill_between(degrees,
                np.array(train_rmse_means)-np.array(train_rmse_stds),
                np.array(train_rmse_means)+np.array(train_rmse_stds),
                alpha=0.15)
ax.fill_between(degrees,
                np.array(test_rmse_means)-np.array(test_rmse_stds),
                np.array(test_rmse_means)+np.array(test_rmse_stds),
                alpha=0.15)
ax.axvline(best_degree, color='gray', linestyle='--', label=f'best degree={best_degree}')
ax.set_title('Repeated split error curves by polynomial degree')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('RMSE')
ax.legend()
plt.show()


In [ ]:
complexity = np.linspace(1, 10, 200)
bias2 = 1.8 / complexity
variance = 0.06 * (complexity - 1.0) ** 2
total = bias2 + variance + 0.08

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(complexity, bias2, label='Bias$^2$')
ax.plot(complexity, variance, label='Variance')
ax.plot(complexity, total, lw=2.5, label='Total expected error')
ax.set_title('Conceptual bias-variance tradeoff')
ax.set_xlabel('Model complexity')
ax.set_ylabel('Error (conceptual scale)')
ax.legend()
plt.show()


## 3) Learning-rate / gradient-descent demonstration (linear regression)

In [ ]:
X_train_gd, X_test_gd, y_train_gd, y_test_gd = train_test_split(
    X.values, y.values, test_size=0.3, random_state=42
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train_s = x_scaler.fit_transform(X_train_gd)
X_test_s = x_scaler.transform(X_test_gd)
y_train_s = y_scaler.fit_transform(y_train_gd.reshape(-1, 1)).ravel()
y_test_s = y_scaler.transform(y_test_gd.reshape(-1, 1)).ravel()

learning_rates = [0.0005, 0.005, 0.05]
max_epochs = 120

loss_histories = {}
for lr in learning_rates:
    sgd = SGDRegressor(
        loss='squared_error',
        learning_rate='constant',
        eta0=lr,
        max_iter=1,
        tol=None,
        penalty=None,
        random_state=42,
        warm_start=True
    )
    losses = []
    for _ in range(max_epochs):
        sgd.fit(X_train_s, y_train_s)
        pred_train = sgd.predict(X_train_s)
        losses.append(mean_squared_error(y_train_s, pred_train))
    loss_histories[lr] = losses

fig, ax = plt.subplots(figsize=(9, 4.8))
for lr, losses in loss_histories.items():
    ax.plot(losses, label=f'eta0={lr}')
ax.set_title('SGD training loss vs epoch for different learning rates')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (scaled target)')
ax.legend()
plt.show()


## 4) Salary regression model comparison (main task)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

models = {
    'LinearRegression': LinearRegression(),
    'Polynomial(deg=3)+Linear': Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False)),
        ('lin', LinearRegression())
    ]),
    'SVR(RBF, scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(C=10, epsilon=0.1, gamma='scale'))
    ]),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42, max_depth=4),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42),
    'MLPRegressor(scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=3000, random_state=42))
    ])
}

def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

rows = []
fitted_models = {}
for name, model in models.items():
    m = clone(model)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    met = regression_metrics(y_test, pred)
    met['Model'] = name
    rows.append(met)
    fitted_models[name] = m

results_reg = pd.DataFrame(rows).set_index('Model').sort_values('RMSE')
results_reg


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
results_reg['RMSE'].sort_values().plot(kind='bar', ax=ax)
ax.set_title('Test RMSE by regression model')
ax.set_ylabel('RMSE')
ax.set_xlabel('Model')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
x_grid = np.linspace(X['YearsExperience'].min(), X['YearsExperience'].max(), 200).reshape(-1, 1)
selected = ['LinearRegression', 'Polynomial(deg=3)+Linear', 'RandomForestRegressor', 'GradientBoostingRegressor']

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_train['YearsExperience'], y_train, alpha=0.6, label='Train data')
ax.scatter(X_test['YearsExperience'], y_test, alpha=0.8, label='Test data')

for name in selected:
    y_curve = fitted_models[name].predict(x_grid)
    ax.plot(x_grid.ravel(), y_curve, lw=2, label=name)

ax.set_title('Salary prediction curves on YearsExperience')
ax.set_xlabel('YearsExperience')
ax.set_ylabel('Salary')
ax.legend(loc='best')
plt.show()


## 5) Neural-network training visualization (regression)

In [ ]:
scaler_nn = StandardScaler()
X_train_nn = scaler_nn.fit_transform(X_train)
X_test_nn = scaler_nn.transform(X_test)

mlp_reg = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    max_iter=1500,
    random_state=42,
    learning_rate_init=0.01,
    early_stopping=False
)
mlp_reg.fit(X_train_nn, y_train)

pred_nn = mlp_reg.predict(X_test_nn)
print('MLPRegressor test metrics:')
print('MAE :', round(mean_absolute_error(y_test, pred_nn), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, pred_nn)), 2))
print('R2  :', round(r2_score(y_test, pred_nn), 4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mlp_reg.loss_curve_)
ax.set_title('MLPRegressor training loss curve')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
plt.show()


## 6) Optional derived-target classification (from salary)

This is **not** the original business target.
It is an optional educational mapping from regression data to classification:

- `high_salary = 1` if `Salary >= median(Salary)`
- `high_salary = 0` otherwise


In [ ]:
median_salary = df['Salary'].median()
df_cls = df.copy()
df_cls['high_salary'] = (df_cls['Salary'] >= median_salary).astype(int)

X_cls = df_cls[['YearsExperience']]
y_cls = df_cls['high_salary']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=42, stratify=y_cls
)

classifiers = {
    'LogisticRegression(scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'SVC(RBF, scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(C=3, gamma='scale', random_state=42))
    ]),
    'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=4, random_state=42),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=300, random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'MLPClassifier(scaled)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=3000, random_state=42))
    ])
}

cls_rows = []
fitted_cls = {}
for name, model in classifiers.items():
    m = clone(model)
    m.fit(Xc_train, yc_train)
    pred = m.predict(Xc_test)
    cls_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(yc_test, pred),
        'Precision': precision_score(yc_test, pred, zero_division=0),
        'Recall': recall_score(yc_test, pred, zero_division=0),
        'F1': f1_score(yc_test, pred, zero_division=0)
    })
    fitted_cls[name] = m

results_cls = pd.DataFrame(cls_rows).set_index('Model').sort_values('F1', ascending=False)
results_cls


In [ ]:
best_cls_name = results_cls.index[0]
best_cls = fitted_cls[best_cls_name]
pred_best = best_cls.predict(Xc_test)

cm = confusion_matrix(yc_test, pred_best)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['low_salary', 'high_salary']).plot(ax=ax, colorbar=False)
ax.set_title(f'Confusion matrix — {best_cls_name}')
plt.show()


## 7) Notes

- Main objective remains regression (`Salary` prediction).
- Classification section is optional and based on a derived target, not the original target.
- Random seeds were fixed for reproducibility.
